# 09 — Modelo de Deep Learning CNN e ResNet50
## Objetivo
Treinar e comparar dois modelos sobre as matrizes de conectividade funcional derivadas das 17.955 características do atlas CC200:

1. **CNN personalizada**, treinada do zero.
2. **ResNet50 com Transfer Learning**, utilizando pesos pré-treinados no ImageNet e treinando inicialmente apenas a camada de classificação.

## Protocolo
- Carrega `X`, `y`, `ids`, `sites` e `fold_id` produzidos pelo Notebook 07.
- Usa os mesmos folds externos agrupados por centro de aquisição do Notebook 08.
- Confere se os centros de treino e teste não se sobrepõem.
- Não faz busca de hiperparâmetros nem early stopping com o fold externo; o número de épocas é fixado previamente.
- Gera previsões out-of-fold (OOF) para todos os participantes.
- Calcula Accuracy, Balanced Accuracy, Precision, Recall, Specificity, F1-score e AUC-ROC.

## Como as matrizes são reconstruídas
Cada vetor de 17.955 características representa as conexões fora da diagonal de uma matriz simétrica de 190 × 190. O notebook reconstrói essa matriz e define a diagonal como zero. A CNN recebe um canal; a ResNet50 recebe três cópias da mesma matriz, redimensionadas para 224 × 224.

**Limitação metodológica:** uma matriz de conectividade não é uma imagem natural. A adaptação da ResNet50 pré-treinada no ImageNet é exploratória e deve ser descrita como tal. O desempenho é estimado por validação cruzada agrupada por centro, não por um teste final independente.

In [26]:
# 1. Imports, configuração e carregamento do dataset

from pathlib import Path
import sys
import warnings
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    roc_auc_score, balanced_accuracy_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix
)

warnings.filterwarnings("default")
SEED = 42

# Localizar a raiz do projeto
cwd = Path.cwd().resolve()
raiz_projeto = next(
    (p for p in [cwd, *cwd.parents]
     if (p / "config.py").is_file() and (p / "notebooks").is_dir()),
    None
)
if raiz_projeto is None:
    raise FileNotFoundError(
        "Não encontrei a raiz do projeto. Abra o notebook dentro da pasta "
        "'notebooks' do projeto TCC_TDAH."
    )
if str(raiz_projeto) not in sys.path:
    sys.path.insert(0, str(raiz_projeto))

import config

arquivo_dataset = Path(config.ARQ_DATASET)
if not arquivo_dataset.is_file():
    raise FileNotFoundError(
        f"Dataset não encontrado: {arquivo_dataset}. Execute primeiro o Notebook 07."
    )

# O dataset deve ter sido produzido pelo próprio Notebook 07.
with np.load(arquivo_dataset, allow_pickle=True) as dados:
    obrigatorias = {"X", "y", "ids", "sites", "fold_id"}
    ausentes = obrigatorias - set(dados.files)
    if ausentes:
        raise KeyError(f"Chaves ausentes em dataset.npz: {sorted(ausentes)}")
    X = np.asarray(dados["X"], dtype=np.float32)
    y = np.asarray(dados["y"], dtype=np.int64).ravel()
    ids = np.asarray(dados["ids"]).astype(str)
    sites = np.asarray(dados["sites"]).astype(str)
    fold_id = np.asarray(dados["fold_id"], dtype=np.int64).ravel()

if X.ndim != 2:
    raise ValueError(f"X deve ser uma matriz 2D; formato recebido: {X.shape}.")
if not (len(X) == len(y) == len(ids) == len(sites) == len(fold_id)):
    raise ValueError("X, y, ids, sites e fold_id têm quantidades diferentes.")
if not np.isfinite(X).all():
    raise ValueError("X contém NaN ou infinito.")
if set(np.unique(y).tolist()) != {0, 1}:
    raise ValueError(
        f"Esperadas classes 0=controle e 1=TDAH; encontradas {np.unique(y).tolist()}."
    )
if np.any(fold_id < 0):
    raise ValueError("Há participantes sem fold definido. Revise o Notebook 07.")

# Inferir o número de ROIs a partir de n_features = n_rois * (n_rois - 1) / 2.
n_features = X.shape[1]
n_rois = int((1 + np.sqrt(1 + 8 * n_features)) / 2)
if n_rois * (n_rois - 1) // 2 != n_features:
    raise ValueError(
        f"{n_features} características não correspondem ao triângulo superior "
        "estrito de uma matriz quadrada. Verifique a saída do Notebook 07."
    )

SEED = int(getattr(config, "SEED", 42))
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Raiz do projeto:", raiz_projeto)
print("Dataset:", arquivo_dataset)
print("X:", X.shape, "| participantes:", len(y), "| ROIs:", n_rois)
print("Classes (0=controle, 1=TDAH):", pd.Series(y).value_counts().sort_index().to_dict())
print("Centros:", len(np.unique(sites)))
print("Folds externos:", sorted(np.unique(fold_id).tolist()))
print("Dispositivo:", device)


Raiz do projeto: C:\Users\Maria Eduarda\Downloads\TCC_TDAH
Dataset: C:\Users\Maria Eduarda\Downloads\TCC_TDAH\data\processed\dataset.npz
X: (709, 17955) | participantes: 709 | ROIs: 190
Classes (0=controle, 1=TDAH): {0: 453, 1: 256}
Centros: 9
Folds externos: [0, 1, 2, 3, 4]
Dispositivo: cpu


In [27]:
# 2. Reconstruir matrizes 190 × 190 e definir os modelos

# X contém o triângulo superior estrito; preencher os dois lados garante simetria.
tri_i, tri_j = np.triu_indices(n_rois, k=1)
matrizes = np.zeros((len(X), n_rois, n_rois), dtype=np.float32)
matrizes[:, tri_i, tri_j] = X
matrizes[:, tri_j, tri_i] = X
# A diagonal permanece zero porque representa ausência de conexão entre uma ROI e ela mesma.

class CNNConectividade(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.40),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


def calcular_metricas(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc_roc": roc_auc_score(y_true, y_prob),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }


# Hiperparâmetros definidos previamente; sem ajuste usando os folds externos.
EPOCAS_CNN = 50
EPOCAS_RESNET = 15
BATCH_CNN = 32
BATCH_RESNET = 8
LR_CNN = 1e-3
LR_RESNET = 1e-3

print("Matrizes reconstruídas:", matrizes.shape)
print("Épocas CNN:", EPOCAS_CNN, "| Épocas ResNet50:", EPOCAS_RESNET)


Matrizes reconstruídas: (709, 190, 190)
Épocas CNN: 50 | Épocas ResNet50: 15


In [28]:
# 3. Funções de treino e inferência

def criar_resnet50():
    # Requer torchvision compatível com a versão instalada do PyTorch.
    try:
        from torchvision.models import resnet50, ResNet50_Weights
    except Exception as exc:
        raise ImportError(
            "Não foi possível importar torchvision.models. Instale uma versão "
            "compatível de torch e torchvision no ambiente do projeto."
        ) from exc

    try:
        modelo = resnet50(weights=ResNet50_Weights.DEFAULT)
    except Exception as exc:
        raise RuntimeError(
            "Não foi possível carregar os pesos pré-treinados da ResNet50. "
            "Verifique a conexão com a internet ou se os pesos estão em cache."
        ) from exc

    # Congelar a rede pré-treinada e treinar somente a camada final.
    for parametro in modelo.parameters():
        parametro.requires_grad = False

    modelo.fc = nn.Linear(modelo.fc.in_features, 1)
    return modelo


def preparar_batch_resnet(x):
    # Entrada: (batch, 1, n_rois, n_rois); ResNet50 espera 3 canais e 224x224.
    x = x.repeat(1, 3, 1, 1)
    x = F.interpolate(
        x, size=(224, 224), mode="bilinear", align_corners=False
    )

    # Normalização convencional dos pesos ImageNet. A aplicação a matrizes de
    # conectividade é exploratória, pois elas não são imagens naturais.
    mean = torch.tensor(
        [0.485, 0.456, 0.406], device=x.device, dtype=x.dtype
    ).view(1, 3, 1, 1)
    std = torch.tensor(
        [0.229, 0.224, 0.225], device=x.device, dtype=x.dtype
    ).view(1, 3, 1, 1)
    return (x - mean) / std


def treinar_modelo(modelo, X_train, y_train, epocas, batch_size, lr, tipo):
    # X_train: (N, n_rois, n_rois); o modelo recebe um canal.
    tensor_x = torch.as_tensor(
        X_train[:, None, :, :], dtype=torch.float32
    )
    tensor_y = torch.as_tensor(
        np.asarray(y_train).reshape(-1), dtype=torch.float32
    )

    loader = DataLoader(
        TensorDataset(tensor_x, tensor_y),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    modelo = modelo.to(device)
    n_neg = int((np.asarray(y_train) == 0).sum())
    n_pos = int((np.asarray(y_train) == 1).sum())
    if n_neg == 0 or n_pos == 0:
        raise ValueError("O conjunto de treino precisa conter as duas classes.")

    peso_pos = torch.tensor(
        [n_neg / n_pos], dtype=torch.float32, device=device
    )
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=peso_pos)

    parametros = [p for p in modelo.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(parametros, lr=lr, weight_decay=1e-4)
    losses = []

    for epoca in range(epocas):
        modelo.train()
        if tipo == "resnet":
            # Mantém a ResNet congelada em modo de avaliação; a camada fc
            # continua treinável porque seus parâmetros requerem gradiente.
            modelo.eval()

        perda_epoca = 0.0
        n_vistos = 0

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device).float().reshape(-1)

            if tipo == "resnet":
                xb = preparar_batch_resnet(xb)

            optimizer.zero_grad(set_to_none=True)
            logits = modelo(xb).reshape(-1)

            # BCEWithLogitsLoss exige logits e rótulos com o mesmo formato.
            if logits.shape != yb.shape:
                raise ValueError(
                    f"Formato incompatível: logits={tuple(logits.shape)}, "
                    f"rótulos={tuple(yb.shape)}."
                )

            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()

            tamanho_lote = len(yb)
            perda_epoca += loss.item() * tamanho_lote
            n_vistos += tamanho_lote

        perda_media = perda_epoca / max(n_vistos, 1)
        losses.append(perda_media)

        if (epoca + 1) % 5 == 0 or epoca == 0:
            print(
                f"    época {epoca + 1:02d}/{epocas} "
                f"| loss={perda_media:.4f}"
            )

    return modelo, losses


@torch.no_grad()
def prever_probabilidades(modelo, X, tipo, batch_size):
    modelo.eval()
    tensor_x = torch.as_tensor(
        np.asarray(X)[:, None, :, :], dtype=torch.float32
    )
    loader = DataLoader(
        TensorDataset(tensor_x),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False
    )

    probabilidades = []
    for (xb,) in loader:
        xb = xb.to(device)

        if tipo == "resnet":
            xb = preparar_batch_resnet(xb)

        logits = modelo(xb).reshape(-1)
        probs = torch.sigmoid(logits)
        probabilidades.extend(probs.cpu().numpy().tolist())

    return np.asarray(probabilidades, dtype=np.float32).reshape(-1)


In [ ]:
# 4. Avaliação externa agrupada por centro: CNN e ResNet50

folds = sorted(np.unique(fold_id).tolist())
nomes_modelos = ["CNN", "ResNet50"]
pred_oof = {nome: np.full(len(y), -1, dtype=np.int8) for nome in nomes_modelos}
proba_oof = {nome: np.full(len(y), np.nan, dtype=np.float32) for nome in nomes_modelos}
resultados_fold = []
historicos_loss = {"CNN": {}, "ResNet50": {}}

pasta_modelos = raiz_projeto / "modelos" / "deep_learning_folds"
pasta_modelos.mkdir(parents=True, exist_ok=True)

for fold in folds:
    print(f"\n{'='*24} FOLD {fold} {'='*24}")
    idx_teste = np.where(fold_id == fold)[0]
    idx_treino = np.where(fold_id != fold)[0]

    # Verificação obrigatória de separação por centro.
    centros_treino = set(sites[idx_treino])
    centros_teste = set(sites[idx_teste])
    sobreposicao = centros_treino.intersection(centros_teste)
    if sobreposicao:
        raise ValueError(
            f"Fold {fold}: centros presentes em treino e teste: {sorted(sobreposicao)}"
        )
    if set(np.unique(y[idx_teste]).tolist()) != {0, 1}:
        raise ValueError(
            f"Fold {fold} não contém as duas classes; AUC-ROC não pode ser calculada "
            "corretamente para esse fold. Revise os folds no Notebook 07."
        )

    Xtr, Xte = matrizes[idx_treino], matrizes[idx_teste]
    ytr, yte = y[idx_treino], y[idx_teste]
    print(
        f"Treino: {len(idx_treino)} participantes / {len(centros_treino)} centros; "
        f"teste: {len(idx_teste)} participantes / {len(centros_teste)} centros"
    )

    # CNN personalizada
    print("\nTreinando CNN personalizada")
    torch.manual_seed(SEED + int(fold))
    np.random.seed(SEED + int(fold))
    cnn = CNNConectividade()
    cnn, loss_cnn = treinar_modelo(
        cnn, Xtr, ytr, EPOCAS_CNN, BATCH_CNN, LR_CNN, tipo="cnn"
    )
    p_cnn = prever_probabilidades(cnn, Xte, tipo="cnn", batch_size=BATCH_CNN)
    pred_cnn = (p_cnn >= 0.5).astype(np.int8)
    pred_oof["CNN"][idx_teste] = pred_cnn
    proba_oof["CNN"][idx_teste] = p_cnn
    m_cnn = calcular_metricas(yte, pred_cnn, p_cnn)
    resultados_fold.append({
        "modelo": "CNN", "fold": int(fold), "n_teste": int(len(idx_teste)), **m_cnn
    })
    historicos_loss["CNN"][int(fold)] = loss_cnn
    torch.save(cnn.state_dict(), pasta_modelos / f"cnn_fold_{fold}.pt")
    print(
        f"CNN | Accuracy={m_cnn['accuracy']:.3f} | F1={m_cnn['f1']:.3f} "
        f"| AUC-ROC={m_cnn['auc_roc']:.3f}"
    )
    del cnn
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ResNet50 pré-treinada: treinar somente a camada final.
    print("\nTreinando ResNet50 (Transfer Learning)")
    torch.manual_seed(SEED + int(fold))
    np.random.seed(SEED + int(fold))
    resnet = criar_resnet50()
    resnet, loss_resnet = treinar_modelo(
        resnet, Xtr, ytr, EPOCAS_RESNET, BATCH_RESNET, LR_RESNET, tipo="resnet"
    )
    p_resnet = prever_probabilidades(
        resnet, Xte, tipo="resnet", batch_size=BATCH_RESNET
    )
    pred_resnet = (p_resnet >= 0.5).astype(np.int8)
    pred_oof["ResNet50"][idx_teste] = pred_resnet
    proba_oof["ResNet50"][idx_teste] = p_resnet
    m_resnet = calcular_metricas(yte, pred_resnet, p_resnet)
    resultados_fold.append({
        "modelo": "ResNet50", "fold": int(fold), "n_teste": int(len(idx_teste)), **m_resnet
    })
    historicos_loss["ResNet50"][int(fold)] = loss_resnet
    torch.save(resnet.state_dict(), pasta_modelos / f"resnet50_fold_{fold}.pt")
    print(
        f"ResNet50 | Accuracy={m_resnet['accuracy']:.3f} | F1={m_resnet['f1']:.3f} "
        f"| AUC-ROC={m_resnet['auc_roc']:.3f}"
    )
    del resnet
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Verificar previsões OOF completas.
for nome in nomes_modelos:
    if np.any(pred_oof[nome] < 0) or not np.isfinite(proba_oof[nome]).all():
        raise RuntimeError(f"Há previsões OOF ausentes para {nome}.")

res_fold = pd.DataFrame(resultados_fold)
display(res_fold.sort_values(["modelo", "fold"]).round(4))



======================== FOLD 0 ========================
Treino: 590 participantes / 7 centros; teste: 119 participantes / 2 centros

Treinando CNN personalizada
    época 01/50 | loss=0.8655
    época 05/50 | loss=0.8557
    época 10/50 | loss=0.8521
    época 15/50 | loss=0.8413
    época 20/50 | loss=0.8394
    época 25/50 | loss=0.8411
    época 30/50 | loss=0.8323
    época 35/50 | loss=0.8295
    época 40/50 | loss=0.8103
    época 45/50 | loss=0.7732
    época 50/50 | loss=0.8035
CNN | Accuracy=0.714 | F1=0.000 | AUC-ROC=0.382

Treinando ResNet50 (Transfer Learning)
    época 01/15 | loss=0.8949
    época 05/15 | loss=0.8234
    época 10/15 | loss=0.7863
    época 15/15 | loss=0.7626
ResNet50 | Accuracy=0.546 | F1=0.449 | AUC-ROC=0.518

======================== FOLD 1 ========================
Treino: 587 participantes / 7 centros; teste: 122 participantes / 2 centros

Treinando CNN personalizada
    época 01/50 | loss=0.8837
    época 05/50 | loss=0.8738
    época 10/50 | loss=